### 코랩에서 한글 사용하기
- 한글 글꼴 설치 => [런타임]-[런타임 다시 시작] 클릭


### 서울시 유가정보(11개 구) 분서

#### [미션1] 서울시 유가정보 가져와 1개로 합치기
1. "/content" 폴더에 "data" 폴더 생성후 파이 업로드
2. data 폴더에 있는 데이터 읽어와 oildf에 모두 저장
3. 데이터는 concat()을 이용해 데이터의 아래에 추가
4. encoding: cp949 또는 utf8이 같이 있어 에러 발생
        try: ~ except: ~ 로 해결



- data 폴더에 있는 파일명/하위폴더명 가져오기: os모듈
- for문을 이용 첫번째 파일부터 순차적으로 읽어와 oildf에 concat 진행
- 파일을 읽을시 encoding 에러
        try:
            utf-8 형식 읽어오기
        except:
            cp949 형식 읽어오기
- 빈 DataFrame 변수 생성
        oildf=pd.DataFrame()

In [ ]:
import pandas as pd
import os

fnames=os.listdir("/content/data") #해당 경로에 있는 파일을 정의

oildf= pd.DataFrame()
for fname in fnames: # for 문 안에 있는 조건에 따라
    try:
        df0=pd.read_csv("/content/data/" + fname, header=2) # 앞서 정의한 경로, 폴더에 있는 파일 다 읽어오는 부분
    except:
        df0=pd.read_csv("/content/data/" + fname, header=2, encoding="cp949") # utf8과 cp949 있는 부분 모두 고려

    oildf=pd.concat([oildf, df0])  # oildf에 읽어온 데이터를 합쳐라. 옵션: ignore_index=True : index 재설정

del df0 # 메모리 관리 차원에서 기존 df0 삭제

oildf.reset_index(drop=True, inplace=True)  # 인덱스 재설정 (오브젝트에서 숫자형으로)

display(oildf.head(3))
oildf.info()

#### [미션2] 데이터 전처리
- "구" 필드 생성하기
- 데이터 출력 필드: 상호, 상표, 구, 세프여부, 휘발유, 경우 만 출력
- NaN 값 포함된 데이터 전체 삭제
- "휘발유", "경유": float64로 변경
- "휘발유", "경유" 데이터의 "-"를 평균값으로 수정

필드별 NaN 객수 확인

In [ ]:
oildf.isna().sum()

구 필드 생성하기

In [ ]:
## loc[] 함수를 이용해 주소에서 구 추출하기
gutxt = []
for i in oildf.index:
    gutxt.append( oildf.loc[i, '주소'].split(" ")[1] )

In [ ]:
# 한줄쓰기로 표현
gutxt=[ oildf.loc[i, '주소'].split(" ")[1] for i in oildf.index ]

In [ ]:
# iloc[] 함수를 이용해 주소에서 구 추출하기
gutxt=[ oildf.iloc[i, 2].split(" ")[1] for i in range(len(oildf)) ]

In [ ]:
# "구" 필드 추가
oildf["구"]=gutxt

상호', '상표', "구", '셀프여부', '휘발유', '경유' 필드만 출력

In [ ]:
# 필드명으로 DataFrame 생성하기
oildf1=oildf[['상호','상표',"구",'셀프여부', '휘발유', '경유']]

In [ ]:
# iloc를 이용한 DataFrame 생성하기
oildf2=oildf.iloc[:, [1, 3,-1, 5, 7, 8]]
oildf2.dtypes

In [ ]:
display(oildf1.column.head(1))

NaN / "-" 값 0으로 변경하기

In [ ]:
# NaN값 0으로 변경
oildf2=oildf2.fillna(0)

In [ ]:
# 휘발유/경우 데이터에서 "-" 값만 찾아 0으로 변경
oildf2.loc[oildf2.휘발유=="-",'휘발유']=0
oildf2.loc[oildf2["경유"]=="-",'경유']=0

In [ ]:
# 고유값 확인하기
oildf2['휘발유'].unique()

데이터형(data type) 변경하기

In [ ]:
# 휘발유/경우 실수형 자료로 변경하기
oildf2=oildf2.astype({"휘발유":float, "경유":float})
oildf2.dtypes  # 데이터 타입 확인하기

휘발유/경유가 0인 값 평균으로 변경하기

In [ ]:
# 휘발유/경유 데이터에서 0인 값을 평균값으로 변경하기
oildf2.loc[oildf2.휘발유==0,'휘발유']=oildf2["휘발유"].mean()
oildf2.loc[oildf2["경유"]==0,'경유']=oildf2["경유"].mean()

In [ ]:
# 0인 값이 인는지 확인
oildf2.loc[oildf2["경유"]==0, :]  # 경유가 0인 데이터 찾아라

### pandas 모듈 이용한 데이터 분석
- groupby : 기준이 되는 필드 값을 기준으로 묶어서 통계 계산

In [ ]:
# oildf2에서 최고가 휘발유 검색
oildf2[oildf2['휘발유']==[oildf2['휘발유']].max() #oildf2의 휘발유 가격과 최대값이 일치하는 부분 검색

In [ ]:
guGroup=oildf2.groupby(by == '구',).mean() # '구' 필드 단위로 평균 값 (숫자만 계산됨)
guGroup.head() # groupby를 쓰면 해당 그룹이 인덱스가 된다. (guGroup 별도 변수 생성됨)

In [ ]:
# 강북구 데이터만 출력
guGroup.loc[['강북구']]

# 강북구, 서대문구 출력
guGroup.loc[['강북구','서대문구']]

guGroup.index # guGroup의 인덱스 확인

In [ ]:
# 강북구 ~ 서초구까지 휘발유 값 출력
guGroup.loc['강북구':'서초구','휘발유']

In [ ]:
# 구별 / 상표별 주유소 개수
oildf2.groupby(["구",'상표'])[["휘발유",'경유']].count() #oildf2라는 데이터에서 '구' 당 주유소 전체 개수

# 구별 / 상표별 주유소 개수 계산 후 by값을 데이터로 변경(reset)
oildf2.groupby(["구",'상표'],as_index=False)["휘발유"].count() # 인덱스를 무시하고 일반 데이터로 만들면서

In [ ]:
# 그룹(구) 안에서 원하는(노원구) 데이터 확인
oildf2.groupby("구").get_group("노원구")

- DataFrame 데이터 저장
  * df.to_csv("경로/파일명.csv", index=True/False, encoding=None)
  * index: True(기본값/index값 저장), False (인덱스 저장 생략)
  * encoding: 기본 UTF8로 필요시 원하는 형식으로 인코딩

In [ ]:
# oildf2 csv 파일로 저장
oildf2.to_csv("서울시 유류정보_0314") #데이터 프레임을 csv로 저장



In [ ]:
# 저장한 파일 다운로드
from google.colab import files

files.download("서울시 유류정보_0314")